# 📘 Fincept Notebook — Moving Average Crossover Backtest

**Trading · Intermediate · ~22 min · pandas + numpy**

The moving-average crossover is the classic trend-following strategy: go long when a fast average crosses above a slow one, and step aside when it crosses back below. In this notebook we generate a reproducible price series, build the signals with pandas, and *backtest* the strategy against simply buying and holding.

**What you'll learn**
- Compute fast/slow simple moving averages with pandas `rolling`
- Turn a crossover into long/flat signals and lag them to avoid look-ahead bias
- Build equity curves and compare strategy vs buy-and-hold
- Report total & annualized return, max drawdown, and a simple Sharpe ratio

> **Requires** `pandas` and `numpy` (bundled with Fincept Terminal's Python environment).


## 1. A reproducible price series

We synthesise 120 trading days of prices with a gentle upward **drift** plus daily **noise**, using `np.random.seed(7)` so the series is identical every run — and identical in every cell, since each cell regenerates it from the same seed. The result is a single random walk that trends up but with realistic pullbacks for the crossover to react to.


In [ ]:
try:
    import numpy as np
    import pandas as pd
except ImportError:
    raise SystemExit("Fincept Notebook needs pandas + numpy — install with: pip install pandas numpy")

def make_prices(n=120, start=100.0, drift=0.0006, vol=0.012, seed=7):
    np.random.seed(seed)
    shocks = np.random.normal(drift, vol, n)
    prices = start * np.cumprod(1 + shocks)
    dates = pd.bdate_range("2024-01-01", periods=n)
    return pd.Series(prices, index=dates, name="close")

px = make_prices()
print("Synthetic daily closes (120 business days)")
print("=" * 44)
print(f"start  {px.iloc[0]:8.2f}")
print(f"end    {px.iloc[-1]:8.2f}")
print(f"min    {px.min():8.2f}")
print(f"max    {px.max():8.2f}")
print()
print("First 5 closes:")
print(px.head().to_string())


## 2. Fast & slow moving averages and the signal

We compute a **fast SMA (10-day)** and a **slow SMA (30-day)** with `Series.rolling(window).mean()`. The position is **long (1)** whenever the fast SMA is above the slow SMA, otherwise **flat (0)**. Critically we `shift(1)` the signal: you can only act on a crossover the *next* day, so trading on today's bar with today's signal would be cheating (look-ahead bias).


In [ ]:
try:
    import numpy as np
    import pandas as pd
except ImportError:
    raise SystemExit("Fincept Notebook needs pandas + numpy — install with: pip install pandas numpy")

def make_prices(n=120, start=100.0, drift=0.0006, vol=0.012, seed=7):
    np.random.seed(seed)
    shocks = np.random.normal(drift, vol, n)
    prices = start * np.cumprod(1 + shocks)
    return pd.Series(prices, index=pd.bdate_range("2024-01-01", periods=n), name="close")

FAST, SLOW = 10, 30
df = make_prices().to_frame()
df["sma_fast"] = df["close"].rolling(FAST).mean()
df["sma_slow"] = df["close"].rolling(SLOW).mean()
df["signal"] = (df["sma_fast"] > df["sma_slow"]).astype(int)
df["position"] = df["signal"].shift(1).fillna(0)  # act next day

trades = int((df["position"].diff().abs() > 0).sum())
print(f"Crossover signals (FAST={FAST}, SLOW={SLOW}) — tail of the table")
print("=" * 60)
print(df.tail(8).round(2).to_string())
print()
print(f"Days long: {int(df['position'].sum())} of {len(df)}   |   position changes: {trades}")


## 3. Equity curves: strategy vs buy-and-hold

The market's daily return is `close.pct_change()`. The strategy only earns that return on days it was positioned long, so `strategy_return = position * market_return`. Compounding each stream gives an **equity curve** (growth of \$1). We compare the crossover strategy with simply buying and holding from day one.


In [ ]:
try:
    import numpy as np
    import pandas as pd
except ImportError:
    raise SystemExit("Fincept Notebook needs pandas + numpy — install with: pip install pandas numpy")

def make_prices(n=120, start=100.0, drift=0.0006, vol=0.012, seed=7):
    np.random.seed(seed)
    shocks = np.random.normal(drift, vol, n)
    prices = start * np.cumprod(1 + shocks)
    return pd.Series(prices, index=pd.bdate_range("2024-01-01", periods=n), name="close")

FAST, SLOW = 10, 30
df = make_prices().to_frame()
df["sma_fast"] = df["close"].rolling(FAST).mean()
df["sma_slow"] = df["close"].rolling(SLOW).mean()
df["position"] = (df["sma_fast"] > df["sma_slow"]).astype(int).shift(1).fillna(0)

df["mkt_ret"] = df["close"].pct_change().fillna(0)
df["strat_ret"] = df["position"] * df["mkt_ret"]
df["equity_hold"] = (1 + df["mkt_ret"]).cumprod()
df["equity_strat"] = (1 + df["strat_ret"]).cumprod()

print("Growth of $1 — last 8 days")
print("=" * 44)
print(df[["close", "position", "equity_hold", "equity_strat"]].tail(8).round(3).to_string())
print()
print(f"Final $1 -> buy & hold : ${df['equity_hold'].iloc[-1]:.3f}")
print(f"Final $1 -> crossover  : ${df['equity_strat'].iloc[-1]:.3f}")


## 4. Performance metrics

Numbers, not vibes. We report for each approach:

- **Total return** — final equity minus 1.
- **Annualized return** — geometric, scaled to 252 trading days.
- **Max drawdown** — the worst peak-to-trough loss along the equity curve.
- **Sharpe ratio** — mean daily return / daily volatility, annualized by √252 (risk-free assumed 0).

Trend-following often trails buy-and-hold in a smoothly rising market because it sits in cash during pullbacks — but it usually suffers a *smaller drawdown*, which is the trade-off this table makes visible.


In [ ]:
try:
    import numpy as np
    import pandas as pd
except ImportError:
    raise SystemExit("Fincept Notebook needs pandas + numpy — install with: pip install pandas numpy")

def make_prices(n=120, start=100.0, drift=0.0006, vol=0.012, seed=7):
    np.random.seed(seed)
    shocks = np.random.normal(drift, vol, n)
    prices = start * np.cumprod(1 + shocks)
    return pd.Series(prices, index=pd.bdate_range("2024-01-01", periods=n), name="close")

FAST, SLOW = 10, 30
df = make_prices().to_frame()
df["sma_fast"] = df["close"].rolling(FAST).mean()
df["sma_slow"] = df["close"].rolling(SLOW).mean()
df["position"] = (df["sma_fast"] > df["sma_slow"]).astype(int).shift(1).fillna(0)
df["mkt_ret"] = df["close"].pct_change().fillna(0)
df["strat_ret"] = df["position"] * df["mkt_ret"]

def max_drawdown(returns):
    equity = (1 + returns).cumprod()
    peak = equity.cummax()
    return ((equity - peak) / peak).min()

def metrics(returns):
    n = len(returns)
    total = (1 + returns).prod() - 1
    ann = (1 + total) ** (252 / n) - 1
    vol = returns.std(ddof=0)
    sharpe = (returns.mean() / vol * np.sqrt(252)) if vol > 0 else float("nan")
    return total, ann, max_drawdown(returns), sharpe

rows = []
for name, col in [("Buy & hold", "mkt_ret"), ("SMA crossover", "strat_ret")]:
    total, ann, mdd, sharpe = metrics(df[col])
    rows.append({
        "Strategy": name,
        "Total %": round(total * 100, 2),
        "Annualized %": round(ann * 100, 2),
        "Max DD %": round(mdd * 100, 2),
        "Sharpe": round(sharpe, 2),
    })

table = pd.DataFrame(rows).set_index("Strategy")
print("Backtest performance metrics")
print("=" * 60)
print(table.to_string())
print()
print("Note: a single random path is not evidence — robust backtesting runs")
print("many paths and includes costs, slippage, and out-of-sample testing.")


---
*— Fincept Notebook · part of Fincept Terminal. Edit any cell and press Ctrl+Enter to run.*
